In [7]:
import os
# os.chdir('../')

In [8]:
import pandas as pd
# import numpy as np
import pickle

# from textwave.modules.extraction.preprocessing import DocumentProcessing
# from textwave.modules.extraction.embedding import Embedding
from textwave.modules.retrieval.index.bruteforce import FaissBruteForce
from textwave.modules.retrieval.search import FaissSearch

from textwave.modules.generator.question_answering import QAGeneratorMistral
from textwave.modules.utils.metrics import Matching

print('DONE')

DONE


In [27]:
QUESTIONS_PATH = 'textwave/qa_resources/question.tsv'
CORPUS_PATH = 'textwave/storage/'
CHUNKING_STRATEGY = 'fixed-length' # 'fixed-length' or 'sentence'
CHUNKING_PARAMETERS = {
    "chunk_size": 150, 
    "overlap_size": 0
}
INDEX_STRATEGY = "bruteforce"
INDEX_PARAMETERS = {
    'metric': 'cosine',
}
K_NEAREST_NEIGHBORS = 3
MISTRAL_MODEL = 'mistral-large-latest'
# mistral-small-latest
# mistral-medium-latest
# mistral-large-latest
API_KEY = os.environ["MISTRAL_API_KEY"]

In [28]:
# Process questions df
raw_questions = pd.read_table(QUESTIONS_PATH)
easy = raw_questions[raw_questions['DifficultyFromAnswerer'] == 'easy'].reset_index(drop=True)[:20]
medium = raw_questions[raw_questions['DifficultyFromAnswerer'] == 'medium'].reset_index(drop=True)[:20]
hard = raw_questions[raw_questions['DifficultyFromAnswerer'] == 'hard'].reset_index(drop=True)[:20]
questions = pd.concat([easy, medium, hard]).reset_index(drop=True)

In [29]:
# Use best performing index
index = FaissBruteForce.load('faiss/bruteforce_cosine_fixed-length_150_0.pkl')

In [30]:
# Load question embeddings
with open('analysis/question_embeddings.pkl', 'rb') as f:
    embeddings_map = pickle.load(f)

In [31]:
# Get unique questions 
unique_questions = questions['Question'].unique()
mistral = QAGeneratorMistral(API_KEY, generator_model=MISTRAL_MODEL)
results = {}
count = 0
for question in unique_questions:
    print(f'{count+1}/{len(unique_questions)}')

    # Embed query
    query_vector = embeddings_map[question]

    # Search index for neighbors
    search = FaissSearch(index, metric=INDEX_PARAMETERS['metric'])
    _, _, meta_results = search.search(query_vector, k=K_NEAREST_NEIGHBORS)

    # Trigger QA object to ping MISTRAL, get reponse, return
    answer = mistral.generate_answer(query=question, context=meta_results)
    results[question] = answer

    # Increment count
    count += 1

results

1/41
2/41
3/41
4/41
5/41
6/41
7/41
8/41
9/41
10/41
11/41
12/41
13/41
14/41
15/41
16/41
17/41
18/41
19/41
20/41
21/41
22/41
23/41
24/41
25/41
26/41
27/41
28/41
29/41
30/41
31/41
32/41
33/41
34/41
35/41
36/41
37/41
38/41
39/41
40/41
41/41


{'Was Abraham Lincoln the sixteenth President of the United States?': 'Yes, Abraham Lincoln was the sixteenth President of the United States. He served from March 4, 1861, until his death on April 15, 1865, and was elected on November 6, 1860.',
 'Did Lincoln sign the National Banking Act of 1863?': 'No context.',
 'Did his mother die of pneumonia?': 'No, his mother did not die of pneumonia. She died of **milk sickness** at the age of thirty-four.',
 "How many long was Lincoln's formal education?": "Lincoln's formal education lasted about **18 months**.",
 'When did Lincoln begin his political career?': 'Abraham Lincoln began his political career in **1832**, at the age of 23.',
 'What did The Legal Tender Act of 1862 establish?': 'The Legal Tender Act of 1862 established the **United States Note**, the first paper currency in the U.S., also known as the **greenback currency**, which was issued during the Civil War and pledged to be redeemed in gold.',
 'Was Abraham Lincoln the first P

In [32]:
# Get metrcis, add to dataframe
# from time import sleep

metrics = Matching()

processed = questions[~questions['Question'].isna()]
processed = processed[~processed['Answer'].isna()]

indices = []
for idx, row in processed.iterrows():
    if row['Question'] not in results:
        indices.append(idx)

processed = processed.drop(indices)

for idx, row in processed.iterrows():
    question = row['Question']
    print(f'{idx+1}/{len(processed)+1}')

    true_answer = row['Answer']
    generated_answer = results[question]

    em = metrics.exact_match(generated_answer, true_answer)
    print(f"Exact Match: {em}")

    try:
        scores, match = metrics.transformer_match(generated_answer, true_answer, question)
        print(f"Transformer Match: {match} | Scores: {scores}\n")
    except:
        print('Failed to get transformer match!')

    processed.at[idx, 'Exact Match'] = em
    processed.at[idx, 'Transformer Match'] = match

Using device: cpu
1/60
Exact Match: True
Transformer Match: True | Scores: {'yes': {'Yes, Abraham Lincoln was the sixteenth President of the United States. He served from March 4, 1861, until his death on April 15, 1865, and was elected on November 6, 1860.': 1.0}}

2/60
Exact Match: True
Transformer Match: True | Scores: {'Yes.': {'Yes, Abraham Lincoln was the sixteenth President of the United States. He served from March 4, 1861, until his death on April 15, 1865, and was elected on November 6, 1860.': 1.0}}

3/60
Exact Match: False
Transformer Match: False | Scores: {'Yes.': {'No context.': 0.05958276}}

4/60
Exact Match: True
Transformer Match: True | Scores: {'No.': {'No, his mother did not die of pneumonia. She died of **milk sickness** at the age of thirty-four.': 1.0}}

5/60
Exact Match: True
Transformer Match: True | Scores: {'18 months': {"Lincoln's formal education lasted about **18 months**.": 1.0}}

6/60
Exact Match: True
Transformer Match: True | Scores: {'1832': {'Abraha

In [33]:
n = processed[~processed['Exact Match'].isna()]
easy = n[n['DifficultyFromQuestioner'] == 'easy']
medium = n[n['DifficultyFromQuestioner'] == 'medium']
hard = n[n['DifficultyFromQuestioner'] == 'hard']

dfs = [easy, medium, hard]

for df in dfs:
    print(len(df[df['Exact Match']==True]) / len(df))
    print(len(df[df['Transformer Match']==True]) / len(df))

0.75
0.75
0.625
0.75
0.35
0.4


In [ ]:
from sentence_transformers import CrossEncoder
model = CrossEncoder(
    "zli12321/answer_equivalence_distilbert",
    num_labels=2
)

config.json:   0%|          | 0.00/478 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [1]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("cuDNN version:", torch.backends.cudnn.version())

PyTorch: 2.2.2
CUDA available: False
CUDA version: None
cuDNN version: None


In [2]:
torch.backends.mps.is_available()

True